In [5]:
import os, sys
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from skimage import img_as_float
import pandas as pd
import matplotlib.patches as mpatches
from matplotlib.widgets import RectangleSelector

import ipywidgets as widgets
from IPython.display import display, clear_output

In [16]:
# List your TIFF z-stacks here (one per channel)
# Order will determine the color mapping you choose later.

# THIS IS WT
path_dir = "/home/data/Shared/shared_datasets/sm_fish/data/sm_fish/082325_Downing_lab"

# THIS IS TKO
#path_dir = "/home/data/Shared/shared_datasets/sm_fish/data/sm_fish/092725_DowningLab/sep20plate/well6"

# names
# 488 is ECAD, not needed
#"647": "GBX2", 
#

wavelengths_to_use = {"555": "NR2F2", "594": "RPL11", "DAPI": "DAPI"}

files = os.listdir(path_dir)

print(files)

records = []

for file in files:

    if '.tif' not in file:
        continue

    path_file = os.path.join(path_dir, file)

    splitname = file.split('-')

    if '_' in file:
        wavelength = splitname[0].split('_')[1]
        replicate = splitname[1].replace('.tif','')
    else:
        wavelength = splitname[1]
        replicate = splitname[2].replace('.tif','')

    if wavelength not in wavelengths_to_use:
        continue

    records.append({'name': file, 'path': path_file, 'replicate': replicate, 'wavelength': wavelength, "name": wavelengths_to_use[wavelength]})

df_files = pd.DataFrame(records)

df_files.sort_values(by=['name', 'replicate'], inplace=True)

print(df_files)


['plate1_DAPI-005.tif', 'plate1_488-001.tif', 'plate1_647-002.tif', 'plate1_647-004.tif', 'plate1_555-002.tif', 'plate1_594-004.tif', 'plate1_555-004.tif', 'plate1_DAPI-002.tif', 'plate1_555-005.tif', 'plate1_DAPI-004.tif', 'plate1_647-001.tif', 'plate1_594-001.tif', 'plate1_594-003.tif', 'plate1_DAPI-001.tif', 'plate1_BF-003.tif', 'plate1_555-003.tif', 'plate1_BF-004.tif', 'plate1_BF-002.tif', 'plate1_488-004.tif', 'plate1_488-002.tif', 'plate1_488-003.tif', 'plate1_555-001.tif', 'plate1_594-002.tif', 'plate1_647-003.tif', 'plate1_594-005.tif', 'plate1_BF-001.tif', 'plate1_488-005.tif', 'plate1_DAPI-003.tif', 'plate1_BF-005.tif', 'plate1_647-005.tif', 'fish_p_594-d-_TO_STACK_zs_AF_Hong_singlePosition.JNL']
     name                                               path replicate  \
9    DAPI  /home/data/Shared/shared_datasets/sm_fish/data...       001   
4    DAPI  /home/data/Shared/shared_datasets/sm_fish/data...       002   
14   DAPI  /home/data/Shared/shared_datasets/sm_fish/data... 

In [11]:
def wavelength_to_rgb(wavelength, gamma=0.8):
    """
    Convert a wavelength (nm) or 'DAPI' string to an RGB tuple in 0–1 range.
    - 'DAPI' → vivid blue
    - Below 380 nm (UV) → violet
    - Above 780 nm (IR) → deep red
    """
    # Special case: DAPI
    if isinstance(wavelength, str) and wavelength.upper() == 'DAPI':
        return (0.2, 0.4, 1.0)  # bright blue

    w = float(wavelength)

    # Pseudocolors for out-of-range
    if w < 380:        # UV → violet
        return (0.6, 0.0, 0.8)
    if w > 780:        # IR → deep red
        return (0.8, 0.0, 0.2)

    # Visible spectrum mapping
    if w < 440:
        R, G, B = -(w - 440) / (440 - 380), 0.0, 1.0
    elif w < 490:
        R, G, B = 0.0, (w - 440) / (490 - 440), 1.0
    elif w < 510:
        R, G, B = 0.0, 1.0, -(w - 510) / (510 - 490)
    elif w < 580:
        # this output greenish but we want max green
        #R, G, B = (w - 510) / (580 - 510), 1.0, 0.0
        R, G, B = 0.0, 1.0, 0.0
    elif w < 645:
        # this output orange but we want a nice red for RPL11
        #R, G, B = 1.0, -(w - 645) / (645 - 580), 0.0
        R, G, B = 1.0, 0.0, 0.0
    else:  # 645–780
        # red?
        R, G, B = 1.0, 0.0, 0.0

    # Edge-of-vision falloff
    if w < 420:
        factor = 0.3 + 0.7 * (w - 380) / (420 - 380)
    elif w > 645:
        factor = 0.3 + 0.7 * (780 - w) / (780 - 645)
    else:
        factor = 1.0

    return tuple((c * factor) ** gamma for c in (R, G, B))


In [14]:
# --- Global storage ---
projections = []
colors = []
names = []
dapi_roi = None

def add_legend(names, colors, ax):
    legend_patches = []
    for name, color in zip(names, colors):
        patch = mpatches.Rectangle((0, 0), 1, 1, facecolor=color, edgecolor='white', linewidth=0.5)
        legend_patches.append(patch)
    
    # Add legend
    legend = ax.legend(legend_patches, names, 
                                 title="Channels",
                                 loc='upper right', 
                                 fontsize=9,
                                 framealpha=0.8,
                                 fancybox=True,
                                 shadow=True)
    
    # Style for dark composite images
    legend.get_frame().set_facecolor('black')
    legend.get_frame().set_edgecolor('white')
    for text in legend.get_texts():
        text.set_color('white')
    legend.get_title().set_color('white')
    legend.get_title().set_weight('bold')

def merge(replicate_name, save_path=None, show_individual=True, show_merged=True, 
          figsize=(15, 5), contrast_enhancement=1.0):
    """
    Merge multiple fluorescence channels into a composite RGB image
    
    Parameters:
    -----------
    replicate_name : str
        Name identifier for this replicate
    save_path : str, optional
        Directory to save the merged images. If None, images are not saved.
    show_individual : bool
        Whether to display individual channels
    show_merged : bool
        Whether to display the merged composite
    figsize : tuple
        Figure size for display
    contrast_enhancement : float
        Factor to enhance contrast (>1 increases contrast)
    """
    global projections, colors, names
    
    if not projections:
        print("No projections to merge!")
        return None
    
    print(f"Merging {len(projections)} channels for {replicate_name}")
    
    # Ensure all projections have the same shape
    shapes = [proj.shape for proj in projections]
    if len(set(shapes)) > 1:
        print(f"Warning: Different shapes detected: {shapes}")
        # Find the minimum dimensions
        min_h = min(shape[0] for shape in shapes)
        min_w = min(shape[1] for shape in shapes)
        projections = [proj[:min_h, :min_w] for proj in projections]
        print(f"Cropped all images to {min_h}x{min_w}")
    
    # Get dimensions
    height, width = projections[0].shape
    
    # Create RGB composite
    composite = np.zeros((height, width, 3))
    
    # Add each channel to the composite
    for i, (proj, color, name) in enumerate(zip(projections, colors, names)):
        # Apply contrast enhancement
        enhanced = np.clip(proj * contrast_enhancement, 0, 1)
        
        # Add to composite with the assigned color
        for c in range(3):
            composite[:, :, c] += enhanced * color[c]
    
    # Normalize composite to prevent oversaturation
    composite = np.clip(composite, 0, 1)
    
    # Display results
    if show_individual or show_merged:
        n_channels = len(projections)
        n_cols = n_channels + (1 if show_merged else 0)
        
        fig, axes = plt.subplots(1, n_cols, figsize=figsize)
        if n_cols == 1:
            axes = [axes]
        
        col_idx = 0
        
        # Show individual channels
        if show_individual:
            for i, (proj, color, name) in enumerate(zip(projections, colors, names)):
                # Create a colored version for display
                colored_channel = np.zeros((height, width, 3))
                enhanced = np.clip(proj * contrast_enhancement, 0, 1)
                for c in range(3):
                    colored_channel[:, :, c] = enhanced * color[c]
                
                axes[col_idx].imshow(colored_channel)
                axes[col_idx].set_title(f'{name}', 
                                       fontsize=10)
                axes[col_idx].axis('off')
                col_idx += 1
        
        # Show merged composite
        if show_merged:
            axes[col_idx].imshow(composite)
            axes[col_idx].set_title(f'Merged Composite\n{replicate_name}', fontsize=12, weight='bold')
            axes[col_idx].axis('off')

            add_legend(names, colors, axes[col_idx])
        
        plt.tight_layout()
        plt.show()
    
    # Save images if requested
    if save_path:
        os.makedirs(save_path, exist_ok=True)
        
        # Save individual channels
        for i, (proj, color, name) in enumerate(zip(projections, colors, names)):
            # Save as grayscale
            plt.figure(figsize=(8, 8))
            plt.imshow(proj, cmap='gray')
            plt.title(f'{name} - {replicate_name}')
            plt.axis('off')
            plt.savefig(os.path.join(save_path, f'{replicate_name}_{name}_grayscale.png'), 
                       dpi=300, bbox_inches='tight')
            plt.close()
            
            # Save as colored
            colored_channel = np.zeros((height, width, 3))
            enhanced = np.clip(proj * contrast_enhancement, 0, 1)
            for c in range(3):
                colored_channel[:, :, c] = enhanced * color[c]
            
            plt.figure(figsize=(8, 8))
            plt.imshow(colored_channel)
            plt.title(f'{name} - {replicate_name}')
            plt.axis('off')
            plt.savefig(os.path.join(save_path, f'{replicate_name}_{name}_colored.png'), 
                       dpi=300, bbox_inches='tight')
            plt.close()
        
        # Save composite
        plt.figure(figsize=(10, 10))
        plt.imshow(composite)
        plt.title(f'Merged Composite - {replicate_name}')
        plt.axis('off')
        #add_legend(names, colors, plt.gca())
        plt.savefig(os.path.join(save_path, f'{replicate_name}_composite.png'), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"Images saved to: {save_path}")
    
    return composite

def reset_processing():
    """Reset global variables for a new processing session"""
    global projections, colors, names, dapi_roi
    projections = []
    colors = []
    names = []
    dapi_roi = None
    print("Processing variables reset. Ready for new session.")

def normalize_array(arr):
    """Normalize array to 0-1 range"""
    arr_norm = arr.copy()
    arr_norm -= arr_norm.min()
    if arr_norm.max() > 0:
        arr_norm /= arr_norm.max()
    return arr_norm

class ImageProcessor:
    def __init__(self, replicate, files):
        self.replicate = replicate
        self.files = files
        self.current_index = 0
        self.main_output = widgets.Output()
        self.process_next()
    
    def process_next(self):
        global dapi_roi
        
        if self.current_index >= len(self.files):
            with self.main_output:
                print("All channels processed. Merging...")
                merge(self.replicate, save_path=f'smfish_{self.replicate}')
            return
        
        record = self.files[self.current_index]
        name = record['name']
        path = record['path']
        wavelength = record['wavelength']
        
        try:
            arr = tiff.imread(path)
        except Exception as e:
            with self.main_output:
                print(f"Error reading {path}: {e}")
            self.current_index += 1
            self.process_next()
            return
        
        with self.main_output:
            clear_output(wait=True)
            print(f"Processing {name} (index {self.current_index})")
        
        if name == 'DAPI':
            self.process_dapi(arr, name, wavelength)
        else:
            self.process_other_channel(arr, name, wavelength)
    
    def process_dapi(self, arr, name, wavelength):
        global dapi_roi
        
        # Clear any previous plots
        plt.close('all')
        
        # Create output for this specific channel
        channel_output = widgets.Output()
        
        # Create sliders
        x_max = max(arr.shape[1], 2)
        y_max = max(arr.shape[0], 2)
        
        x0_slider = widgets.IntSlider(min=0, max=x_max-2, value=0, description='x0')
        x1_slider = widgets.IntSlider(min=2, max=x_max, value=x_max, description='x1')
        y0_slider = widgets.IntSlider(min=0, max=y_max-2, value=0, description='y0')
        y1_slider = widgets.IntSlider(min=2, max=y_max, value=y_max, description='y1')
        
        button = widgets.Button(description=f"Finalize {name} ROI")
        
        def update_preview(*args):
            x0 = max(0, min(x0_slider.value, x1_slider.value - 1))
            x1 = min(arr.shape[1], max(x1_slider.value, x0_slider.value + 1))
            y0 = max(0, min(y0_slider.value, y1_slider.value - 1))
            y1 = min(arr.shape[0], max(y1_slider.value, y0_slider.value + 1))
            
            with channel_output:
                clear_output(wait=True)
                try:
                    cropped = arr[y0:y1, x0:x1]
                    if cropped.size > 0:
                        plt.figure(figsize=(6, 4))
                        plt.imshow(cropped, cmap='gray')
                        plt.title(f'{name} - ROI: ({x0},{y0}) to ({x1},{y1})')
                        plt.axis('off')
                        plt.show()
                        plt.close()
                    else:
                        print("Invalid ROI selected")
                except Exception as e:
                    print(f"Error in preview: {e}")
        
        def on_finalize(*args):
            global dapi_roi
            
            # Remove observers to prevent further callbacks
            for slider in [x0_slider, x1_slider, y0_slider, y1_slider]:
                slider.unobserve_all()
            
            try:
                x0 = max(0, min(x0_slider.value, x1_slider.value - 1))
                x1 = min(arr.shape[1], max(x1_slider.value, x0_slider.value + 1))
                y0 = max(0, min(y0_slider.value, y1_slider.value - 1))
                y1 = min(arr.shape[0], max(y1_slider.value, y0_slider.value + 1))
                
                dapi_roi = (y0, y1, x0, x1)
                
                # Process the cropped DAPI
                cropped = arr[y0:y1, x0:x1].astype(np.float32)
                cropped = normalize_array(cropped)
                
                projections.append(cropped)
                colors.append(wavelength_to_rgb(wavelength))
                names.append(name)
                
                with channel_output:
                    clear_output(wait=True)
                    print(f"{name} ROI finalized. Moving to next channel...")
                
                # Close the channel output and continue
                channel_output.close()
                plt.close('all')
                
                self.current_index += 1
                self.process_next()
                
            except Exception as e:
                with channel_output:
                    clear_output(wait=True)
                    print(f"Error in finalize: {e}")
        
        # Attach observers
        x0_slider.observe(update_preview, 'value')
        x1_slider.observe(update_preview, 'value')
        y0_slider.observe(update_preview, 'value')
        y1_slider.observe(update_preview, 'value')
        
        button.on_click(on_finalize)
        
        # Display in main output
        with self.main_output:
            display(widgets.VBox([
                widgets.HTML(f"<h3>Select ROI for {name}</h3>"),
                x0_slider, x1_slider, y0_slider, y1_slider, 
                button, channel_output
            ]))
        
        # Show initial preview
        update_preview()
    
    def process_other_channel(self, arr, name, wavelength):
        global dapi_roi
        
        # Apply DAPI ROI if available
        if dapi_roi is not None:
            y0, y1, x0, x1 = dapi_roi
            if arr.ndim == 3:
                arr_cropped = arr[:, y0:y1, x0:x1]
            else:
                arr_cropped = arr[y0:y1, x0:x1]
        else:
            arr_cropped = arr
        
        if arr_cropped.ndim == 3:
            self.process_zstack(arr_cropped, name, wavelength)
        elif arr_cropped.ndim == 2:
            # Process 2D image directly
            proc = normalize_array(arr_cropped.astype(np.float32))
            projections.append(proc)
            colors.append(wavelength_to_rgb(wavelength))
            names.append(name)
            
            with self.main_output:
                print(f"{name} processed (2D). Moving to next channel...")
            
            self.current_index += 1
            self.process_next()
        else:
            with self.main_output:
                print(f"Unexpected shape {arr_cropped.shape} for {name}")
            self.current_index += 1
            self.process_next()
    
    def process_zstack(self, arr_cropped, name, wavelength):
        plt.close('all')
        
        # Create output for this channel
        channel_output = widgets.Output()
        
        # Create Z-stack sliders
        z_max = max(arr_cropped.shape[0], 1)
        z_start = widgets.IntSlider(min=0, max=z_max-1, value=0, description='Z start')
        z_end = widgets.IntSlider(min=0, max=z_max-1, value=z_max-1, description='Z end')
        
        # Brightness adjustment sliders (applied before background removal)
        brightness_min = widgets.FloatSlider(
            value=0.0, min=0.0, max=1.0, step=0.01,
            description='Bright Min:',
            style={'description_width': '80px'}
        )
        
        brightness_max = widgets.FloatSlider(
            value=1.0, min=0.0, max=1.0, step=0.01,
            description='Bright Max:',
            style={'description_width': '80px'}
        )
        
        # Background removal controls
        bg_method = widgets.Dropdown(
            options=[('None', 'none'), ('Percentile', 'percentile'), ('Hard Cutoff', 'hard_cutoff'),
                    ('Otsu', 'otsu'), ('Gaussian Mix', 'gaussian_mixture'), ('Adaptive', 'adaptive')],
            value='percentile',
            description='Background:'
        )
        
        percentile_slider = widgets.FloatSlider(
            value=97.0, min=90.0, max=99.9, step=0.1,
            description='Percentile:', 
            style={'description_width': '80px'}
        )
        
        cutoff_slider = widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01,
            description='Cutoff:',
            style={'description_width': '80px'}
        )
        
        sigma_slider = widgets.FloatSlider(
            value=5.0, min=1.0, max=20.0, step=0.5,
            description='Sigma:',
            style={'description_width': '80px'}
        )
        
        # Show/hide parameter controls based on method
        def update_controls(*args):
            
            if bg_method.value == 'percentile':
                percentile_slider.layout.display = 'flex'
                cutoff_slider.layout.display = 'none'
                sigma_slider.layout.display = 'none'
            elif bg_method.value == 'hard_cutoff':
                percentile_slider.layout.display = 'none'
                cutoff_slider.layout.display = 'flex'
                sigma_slider.layout.display = 'none'
            elif bg_method.value == 'adaptive':
                percentile_slider.layout.display = 'none'
                cutoff_slider.layout.display = 'none'
                sigma_slider.layout.display = 'flex'
            else:
                percentile_slider.layout.display = 'none'
                cutoff_slider.layout.display = 'none'
                sigma_slider.layout.display = 'none'
        
        bg_method.observe(update_controls, 'value')
        update_controls()  # Initial setup
        
        button = widgets.Button(description=f"Finalize {name} Z range")
        
        def apply_brightness_adjustment(image, vmin, vmax):
            """Apply brightness adjustment by clipping and rescaling"""
            # Clip to the specified range
            adjusted = np.clip(image, vmin, vmax)
            # Rescale to 0-1
            if vmax > vmin:
                adjusted = (adjusted - vmin) / (vmax - vmin)
            return adjusted
        
        def apply_background_removal(image, method):
            """Apply selected background removal method"""
            if method == 'none':
                return image
            elif method == 'percentile':
                cutoff = np.percentile(image, percentile_slider.value)
                result = np.clip(image - cutoff, 0, 1)
                return result / result.max() if result.max() > 0 else result
            elif method == 'hard_cutoff':
                # Direct intensity threshold - set everything below cutoff to 0
                result = image.copy()
                result[result < cutoff_slider.value] = 0
                # Optionally subtract the cutoff value from remaining pixels
                if cutoff_slider.value > 0:
                    result = np.clip(result - cutoff_slider.value, 0, 1)
                    result = result / result.max() if result.max() > 0 else result
                return result
            elif method == 'otsu':
                from skimage import filters
                threshold = filters.threshold_otsu(image)
                result = np.clip(image - threshold, 0, 1)
                return result / result.max() if result.max() > 0 else result
            elif method == 'gaussian_mixture':
                from sklearn.mixture import GaussianMixture
                pixels = image.flatten().reshape(-1, 1)
                gmm = GaussianMixture(n_components=2, random_state=42)
                gmm.fit(pixels)
                means = gmm.means_.flatten()
                bg_component = np.argmin(means)
                threshold = means[bg_component] + 2 * np.sqrt(gmm.covariances_[bg_component, 0, 0])
                result = np.clip(image - threshold, 0, 1)
                return result / result.max() if result.max() > 0 else result
            elif method == 'adaptive':
                from skimage import filters
                background = filters.gaussian(image, sigma=sigma_slider.value)
                result = np.clip(image - background, 0, 1)
                return result / result.max() if result.max() > 0 else result
            else:
                return image
        
        def update_z_preview(*args):
            start = min(z_start.value, z_end.value)
            end = max(z_start.value, z_end.value)
            
            with channel_output:
                clear_output(wait=True)
                try:
                    if start <= end < arr_cropped.shape[0]:
                        # Create regular max projection
                        regular_proj = arr_cropped[start:end+1].max(axis=0).astype(np.float32)
                        regular_proj = normalize_array(regular_proj)
                        
                        # Apply brightness adjustment BEFORE background removal
                        brightness_adjusted = apply_brightness_adjustment(
                            regular_proj, brightness_min.value, brightness_max.value
                        )
                        
                        # Apply background removal
                        enhanced_proj = apply_background_removal(brightness_adjusted, bg_method.value)
                        
                        # Show comparison
                        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
                        
                        axes[0].imshow(regular_proj, cmap='gray')
                        axes[0].set_title(f'{name} - Original Max Projection')
                        axes[0].axis('off')
                        
                        axes[1].imshow(brightness_adjusted, cmap='gray')
                        axes[1].set_title(f'{name} - Brightness Adjusted')
                        axes[1].axis('off')
                        
                        axes[2].imshow(enhanced_proj, cmap='hot')
                        axes[2].set_title(f'{name} - Background Removed')
                        axes[2].axis('off')
                        
                        # Histogram comparison
                        axes[3].hist(regular_proj.flatten(), bins=50, alpha=0.6, 
                                   label='Original', density=True, color='blue')
                        axes[3].hist(brightness_adjusted.flatten(), bins=50, alpha=0.6, 
                                   label='Brightness Adj', density=True, color='green')
                        if bg_method.value != 'none':
                            axes[3].hist(enhanced_proj.flatten(), bins=50, alpha=0.6, 
                                       label='Enhanced', density=True, color='red')
                        axes[3].set_xlabel('Intensity')
                        axes[3].set_ylabel('Density')
                        axes[3].legend()
                        axes[3].set_title('Intensity Distribution')
                        
                        plt.tight_layout()
                        plt.show()
                        plt.close()
                        
                        # Print stats
                        print(f"Z range: {start} to {end}")
                        print(f"Original - Mean: {np.mean(regular_proj):.4f}, Max: {np.max(regular_proj):.4f}")
                        print(f"Brightness Adjusted - Mean: {np.mean(brightness_adjusted):.4f}, Max: {np.max(brightness_adjusted):.4f}")
                        if bg_method.value != 'none':
                            bright_pixels = np.sum(enhanced_proj > 0.1)
                            print(f"Enhanced - Mean: {np.mean(enhanced_proj):.4f}, Max: {np.max(enhanced_proj):.4f}")
                            print(f"Bright pixels (>0.1): {bright_pixels} ({100*bright_pixels/enhanced_proj.size:.2f}%)")
                        
                    else:
                        print("Invalid Z range")
                except Exception as e:
                    print(f"Error in Z preview: {e}")
        
        def on_z_finalize(*args):
            # Remove observers
            z_start.unobserve_all()
            z_end.unobserve_all()
            bg_method.unobserve_all()
            percentile_slider.unobserve_all()
            sigma_slider.unobserve_all()
            brightness_min.unobserve_all()
            brightness_max.unobserve_all()
            
            try:
                start = min(z_start.value, z_end.value)
                end = max(z_start.value, z_end.value)
                
                # Create regular projection
                regular_proj = arr_cropped[start:end+1].max(axis=0).astype(np.float32)
                regular_proj = normalize_array(regular_proj)
                
                # Apply brightness adjustment BEFORE background removal
                brightness_adjusted = apply_brightness_adjustment(
                    regular_proj, brightness_min.value, brightness_max.value
                )
                
                # Apply background removal
                final_proj = apply_background_removal(brightness_adjusted, bg_method.value)
                
                projections.append(final_proj)
                colors.append(wavelength_to_rgb(wavelength))
                names.append(name)
                
                with channel_output:
                    clear_output(wait=True)
                    method_str = f" ({bg_method.value})" if bg_method.value != 'none' else ""
                    print(f"{name} Z-stack finalized{method_str}. Moving to next channel...")
                
                # Close output and continue
                channel_output.close()
                plt.close('all')
                
                self.current_index += 1
                self.process_next()
                
            except Exception as e:
                with channel_output:
                    clear_output(wait=True)
                    print(f"Error in Z finalize: {e}")
        
        # Attach observers
        z_start.observe(update_z_preview, 'value')
        z_end.observe(update_z_preview, 'value')
        bg_method.observe(update_z_preview, 'value')
        percentile_slider.observe(update_z_preview, 'value')
        sigma_slider.observe(update_z_preview, 'value')
        brightness_min.observe(update_z_preview, 'value')
        brightness_max.observe(update_z_preview, 'value')
        button.on_click(on_z_finalize)
        
        # Create the controls layout
        brightness_controls = widgets.VBox([
            widgets.HTML("<b>Brightness Adjustment:</b>"),
            brightness_min,
            brightness_max
        ])
        
        bg_controls = widgets.VBox([
            widgets.HTML("<b>Background Removal:</b>"),
            bg_method,
            percentile_slider,
            sigma_slider
        ])
        
        # Display in main output
        with self.main_output:
            display(widgets.VBox([
                widgets.HTML(f"<h3>Configure Z-stack, Brightness, and Background for {name}</h3>"),
                widgets.HBox([
                    widgets.VBox([
                        widgets.HTML("<b>Z-stack Range:</b>"),
                        z_start, 
                        z_end,
                        button
                    ]),
                    brightness_controls,
                    bg_controls
                ]),
                channel_output
            ]))
        
        # Show initial preview
        update_z_preview()
    
    def display(self):
        display(self.main_output)

def process_file(files, replicate, index=0):
    """Main entry point - creates and displays the processor"""
    processor = ImageProcessor(replicate, files)
    processor.display()

# Example usage:
# files = [
#     {'name': 'DAPI', 'path': 'path/to/dapi.tif', 'wavelength': 400},
#     {'name': 'GFP', 'path': 'path/to/gfp.tif', 'wavelength': 500},
#     # ... more files
# ]
# process_file(files, index=0)

In [17]:
replicate = '001'

df_filt = df_files.loc[df_files['replicate'] == replicate]

files = df_filt.to_dict('records')

reset_processing()

process_file(files, f'{replicate}', index=0)


Processing variables reset. Ready for new session.


Output()